# Week 8 · Notebook 1  Local Model Playground (Ollama)

**Chat, system prompts, JSON-mode structured triage, and a tokens/sec benchmark  all through the `ollama` CLI via subprocess.**

```
# Requirements: none (uses the installed `ollama` CLI)
```

```
# ⚠️ REQUIRES: Ollama installed (ollama pull llama3.2 or qwen2.5:3b)
```

If Ollama is not installed, every Ollama cell prints a clear message and the notebook still completes (printing a `0.0` metric). Part of AI Engineering Lab · ZoroLogistics case study.

## Why run local

Local inference means zero marginal token cost, no data leaves the machine, and full control over quantization  the right answer for air-gapped triage of customer tickets (the same pattern Zorost's sovereign-AI work uses in production). The tradeoff is VRAM and throughput, which we measure directly below.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (assumes cwd == week-NN/)
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro").is_dir():
        sys.path.insert(0, str(_p)); break

from zoro import data

tickets = data.support_tickets(20, seed=99, n_shipments=10000)
samples = tickets.sample(5, random_state=3)["text"].tolist()
for s in samples:
    print("-", s)

In [ ]:
import subprocess, shutil, os, re, json, time

OLLAMA = shutil.which("ollama") is not None
installed = []
if OLLAMA:
    try:
        r = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=60)
        print("ollama list:\n" + r.stdout)
        for line in r.stdout.splitlines()[1:]:
            parts = line.split()
            if parts:
                installed.append(parts[0])
    except Exception as e:
        print("ollama list failed:", e)
else:
    print("⚠️ Ollama not found. Install from https://ollama.com, then run: ollama pull llama3.2")
    print("   Ollama cells are skipped; the notebook still completes and prints a 0 metric.")

preference = ["llama3.2", "qwen2.5:3b", "llama3.2:3b", "qwen2.5:7b"]
primary = next((m for m in preference if m in installed), (installed[0] if installed else None))
print("primary model:", primary)

## Basic chat via the CLI

`ollama run MODEL "prompt"` is the simplest possible call  one-shot completion through the subprocess boundary.

In [ ]:
def ollama_run(model, prompt, flags=None, timeout=300):
    cmd = ["ollama", "run", model] + (flags or []) + [prompt]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return r.stdout.strip(), r.stderr.strip()

if OLLAMA and primary:
    out, err = ollama_run(primary, "Say hello and confirm you are running locally, in one sentence.")
    print("chat output:", out)
else:
    print("⚠️ Ollama unavailable, skipping chat.")

## System prompt + JSON-mode structured triage

A **Modelfile** bakes in a system prompt and parameters (`temperature 0`); `--format json` constrains the output to JSON. The structured-output rule still applies locally: *a schema constrains shape, never truth*  so we parse, then treat the result as data.

In [ ]:
def triage_prompt(ticket):
    return ("You are a ZoroLogistics support triage agent. Classify the ticket below and return a single JSON object "
            "with keys: category (tracking|damage|refund|documents|customs|billing), priority (low|medium|high|critical), "
            "and summary (one short sentence).\n\nTicket: " + ticket)

ticket_text = samples[0]

if OLLAMA and primary:
    import tempfile
    modelfile = ("FROM " + primary + "\n"
                 'SYSTEM You are a ZoroLogistics support triage assistant. Always answer with a single JSON object with keys "category", "priority", "summary".\n'
                 "PARAMETER temperature 0\n")
    with tempfile.NamedTemporaryFile("w", suffix=".Modelfile", delete=False) as f:
        f.write(modelfile)
        mf_path = f.name
    subprocess.run(["ollama", "create", "zoro-triage", "-f", mf_path], capture_output=True, text=True, timeout=300)
    out, err = ollama_run("zoro-triage", triage_prompt(ticket_text), flags=["--format", "json"])
    print("raw JSON-mode output:", out)
    try:
        parsed = json.loads(out)
        print("parsed triage:", parsed)
    except Exception:
        print("(output was not parseable JSON, inspect the raw text above)")
else:
    print("⚠️ Ollama unavailable, skipping structured triage.")

## Tokens/sec timing loop

`ollama run --verbose` reports an `eval rate` (generation tokens/second). We run the same prompt a few times and average  the number that decides whether a model is usable interactively.

In [ ]:
def eval_rate(stderr):
    m = re.search(r"eval rate:\s*([\d.]+)\s*tokens/s", stderr)
    return float(m.group(1)) if m else None

def benchmark(model, prompt, n=3, json_mode=False):
    flags = ["--verbose"]
    if json_mode:
        flags += ["--format", "json"]
    rates = []
    for _ in range(n):
        subprocess.run(["ollama", "run", model] + flags + [prompt],
                       capture_output=True, text=True, timeout=300) # warm-up
        r = subprocess.run(["ollama", "run", model] + flags + [prompt],
                           capture_output=True, text=True, timeout=300)
        rates.append(eval_rate(r.stderr))
    return rates

if OLLAMA and primary:
    rates = benchmark(primary, "Summarize this shipment delay in one sentence: a severe storm held the truck for six hours.", n=3)
    print("eval rates (tokens/s):", rates)
    tokens_per_sec = sum(r for r in rates if r) / max(1, sum(1 for r in rates if r))
    print(f"primary {primary}: {tokens_per_sec:.1f} tokens/s")
else:
    tokens_per_sec = 0.0
    print("⚠️ Ollama unavailable, tokens/s set to 0.")

## Model comparison table

Run the same triage prompt across the installed models (up to 4), recording tokens/sec, whether JSON-mode output parsed, and output length  the beginning of the backend benchmark table this week's use case asks for.

In [ ]:
import pandas as pd

rows = []
if OLLAMA and installed:
    for m in installed[:4]:
        out, err = ollama_run(m, triage_prompt(ticket_text), flags=["--format", "json"])
        try:
            json.loads(out); ok = True
        except Exception:
            ok = False
        rates = benchmark(m, "Classify this ticket in one word: my shipment arrived damaged.", n=2)
        rate = sum(r for r in rates if r) / max(1, sum(1 for r in rates if r))
        rows.append({"model": m, "tokens_per_sec": round(rate, 1), "json_ok": ok, "output_len": len(out)})
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("⚠️ no models installed, comparison table empty.")

In [ ]:
# Week 8 · Notebook 1 headline metric: mean generation tokens/sec for the primary local model.
print("WEEK8_NB1_TOKENS_PER_SEC:", round(tokens_per_sec, 2))